# EDA: Assignment 1 part 2

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import numpy as np
import statsmodels.api as sm

In [2]:
folder_name = "BaselineData_V2"
file_name = "DataBaseline_v2.xlsx"

In [3]:
file_path = Path.cwd() / folder_name / file_name

In [4]:
data = pd.read_excel(file_path, sheet_name="Data")


In [5]:
data.describe

<bound method NDFrame.describe of        Blocks interview__key  Longitude  Latitude  Dwelling_count  \
0         1.0    00-00-57-72   28.04887 -26.18302           593.0   
1         1.0    00-00-57-72   28.04887 -26.18302           593.0   
2         1.0    00-00-57-72   28.04887 -26.18302           593.0   
3         3.0    00-01-36-83   28.04576 -26.18803          1641.0   
4         3.0    00-01-36-83   28.04576 -26.18803          1641.0   
...       ...            ...        ...       ...             ...   
46210     5.0    99-95-76-20   28.05184 -26.20089          1823.0   
46211     5.0    99-95-76-20   28.05184 -26.20089          1823.0   
46212     5.0    99-95-76-20   28.05184 -26.20089          1823.0   
46213     1.0    99-96-54-48   28.04749 -26.19411           776.0   
46214     1.0    99-96-54-48   28.04749 -26.19411           776.0   

       hhr_register_roster__id  register_members_roster__id Institution  \
0                            1                            1   

In [6]:
melusi_df = data[data["sites"].str.lower() == "melusi"]


site_counts = melusi_df["sites"].value_counts().reset_index()
site_counts.columns = ["Site", "Count"]

In [7]:
melusi_df = melusi_df[
    [
        "Blocks",
        "interview__key",
        "Longitude",
        "Latitude",
        "Dwelling_count",
        "hhr_register_roster__id",
        "register_members_roster__id",
        "hhr_member_total"
    ]
]


melusi_df.head()

,Blocks,interview__key,Longitude,Latitude,Dwelling_count,hhr_register_roster__id,register_members_roster__id,hhr_member_total
19766,11.0,00-06-09-86,28.12639,-25.72485,1007.0,1,1,1
19767,11.0,00-06-09-86,28.12639,-25.72485,1007.0,1,2,1
19768,12.0,00-06-28-22,28.12148,-25.72149,1655.0,1,1,4
19769,12.0,00-06-28-22,28.12148,-25.72149,1655.0,1,2,4
19770,12.0,00-06-28-22,28.12148,-25.72149,1655.0,1,3,4


In [8]:
melusi_df.to_csv("melusi_filtered.csv", index=False)

In [21]:
df = pd.read_csv("melusi_filtered.csv")

df.head()

,Blocks,interview__key,Longitude,Latitude,Dwelling_count,hhr_register_roster__id,register_members_roster__id,hhr_member_total
0,11.0,00-06-09-86,28.12639,-25.72485,1007.0,1,1,1
1,11.0,00-06-09-86,28.12639,-25.72485,1007.0,1,2,1
2,12.0,00-06-28-22,28.12148,-25.72149,1655.0,1,1,4
3,12.0,00-06-28-22,28.12148,-25.72149,1655.0,1,2,4
4,12.0,00-06-28-22,28.12148,-25.72149,1655.0,1,3,4


In [22]:
df = df.drop_duplicates()

In [23]:
df["household_id"] = (
    df["interview__key"].astype(str) + "_" +
    df["hhr_register_roster__id"].astype(str)
)

In [24]:
household_clean = (
    df.groupby(["interview__key", "hhr_register_roster__id"], as_index=False)
      .agg(
          Blocks=("Blocks", "first"),
          Longitude=("Longitude", "first"),
          Latitude=("Latitude", "first"),
          Dwelling_count=("Dwelling_count", "first"),
          corrected_population=("register_members_roster__id", "nunique")
      )
)

household_clean.head()

,interview__key,hhr_register_roster__id,Blocks,Longitude,Latitude,Dwelling_count,corrected_population
0,00-06-09-86,1,11.0,28.12639,-25.72485,1007.0,2
1,00-06-28-22,1,12.0,28.12148,-25.72149,1655.0,4
2,00-07-01-40,1,4.0,28.12675,-25.72045,619.0,2
3,00-16-49-32,1,3.0,28.11343,-25.71910,1345.0,2
4,00-17-50-09,1,2.0,28.10620,-25.72414,735.0,2


In [25]:
household_clean.to_csv("melusi_population_clean.csv", index=False)

In [26]:
df = pd.read_csv("melusi_population_clean.csv")

df.head()

,interview__key,hhr_register_roster__id,Blocks,Longitude,Latitude,Dwelling_count,corrected_population
0,00-06-09-86,1,11.0,28.12639,-25.72485,1007.0,2
1,00-06-28-22,1,12.0,28.12148,-25.72149,1655.0,4
2,00-07-01-40,1,4.0,28.12675,-25.72045,619.0,2
3,00-16-49-32,1,3.0,28.11343,-25.71910,1345.0,2
4,00-17-50-09,1,2.0,28.10620,-25.72414,735.0,2


In [28]:
block_population = df.groupby("Blocks")["corrected_population"].sum().reset_index()


fig = px.bar(
    block_population,
    x="Blocks",
    y="corrected_population",
    title="Total Population per Block in Melusi",
    labels={"corrected_population": "Total Population", "Blocks": "Block"},
)

fig.show()

In [29]:
block_avg = df.groupby("Blocks")["corrected_population"].mean().reset_index()

fig = px.bar(
    block_avg,
    x="Blocks",
    y="corrected_population",
    title="Average Household Size per Block",
    labels={"corrected_population": "Average Household Size"}
)

fig.show()

In [30]:
fig = px.box(
    df,
    x="Blocks",
    y="corrected_population",
    title="Distribution of Household Size by Block",
    labels={"corrected_population": "Household Members"}
)

fig.show()

In [31]:
fig = px.scatter_mapbox(
    df,
    lat="Latitude",
    lon="Longitude",
    color="corrected_population",
    size="corrected_population",
    hover_name="Blocks",
    zoom=14,
    title="Household Population Distribution in Melusi based on the Survey"
)

fig.update_layout(mapbox_style="open-street-map")

fig.show()

/var/folders/7h/t0f4h_k54vv9j6759pcclxgw0000gn/T/ipykernel_29748/825211352.py:1: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [32]:
fig = px.histogram(
    df,
    x="corrected_population",
    nbins=20,
    title="Distribution of Household Size",
    labels={"corrected_population": "Number of Household Members"}
)

fig.show()